# Arecanut Leaf Disease Detection using Hybrid QNN

This notebook contains:

- Data loading
- Model architecture
- Training pipeline
- Accuracy evaluation

Final Model Accuracy: ~96%

In [ ]:
!pip install pennylane

In [ ]:
import os
import torch
import pennylane as qml
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
!cp -r /content/drive/MyDrive/dataset /content/

In [ ]:
train_dir = "/content/dataset/train"
test_dir = "/content/dataset/test"

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

In [ ]:
train_dataset = torchvision.datasets.ImageFolder(
    train_dir,
    transform=transform
)

test_dataset = torchvision.datasets.ImageFolder(
    test_dir,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

In [ ]:
n_qubits = 4

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_circuit(inputs, weights):

    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    qml.templates.StronglyEntanglingLayers(
        weights,
        wires=range(n_qubits)
    )

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [ ]:
weight_shapes = {"weights": (3, n_qubits, 3)}

qnn = qml.qnn.TorchLayer(
    quantum_circuit,
    weight_shapes
)

In [ ]:
class HybridModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3,16,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16,32,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1,1))
        )

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(32,4)

        self.qnn = qnn

        self.fc2 = nn.Linear(4,2)

    def forward(self,x):

        x = self.conv(x)
        x = self.flatten(x)
        x = self.fc1(x)

        x = x.float()

        # QNN batch workaround
        qnn_outputs = []

        for i in range(x.shape[0]):
            q_out = self.qnn(x[i])
            qnn_outputs.append(q_out)

        x = torch.stack(qnn_outputs)

        x = self.fc2(x)

        return x

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = HybridModel().to(device)

In [ ]:
healthy = 605
yellow = 1477

class_weights = torch.tensor([
    yellow/healthy,
    1.0
]).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
def calculate_accuracy(loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [ ]:
checkpoint_path = "/content/drive/MyDrive/arecanut_qnn.pth"

start_epoch = 0
best_acc = 0

if os.path.exists(checkpoint_path):

    print("Loading checkpoint...")

    checkpoint = torch.load(checkpoint_path)

    model.load_state_dict(checkpoint['model'])

    optimizer.load_state_dict(checkpoint['optimizer'])

    start_epoch = checkpoint['epoch']
    best_acc = checkpoint.get('best_acc', 0)

In [ ]:
epochs = 40

for epoch in range(start_epoch, epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()


    train_acc = calculate_accuracy(train_loader)
    test_acc = calculate_accuracy(test_loader)

    if test_acc > best_acc:
      best_acc = test_acc
      torch.save(model.state_dict(),
                "/content/drive/MyDrive/best_arecanut_qnn.pth")

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {total_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Test Accuracy: {test_acc:.2f}%")

    # Save checkpoint
    torch.save({
        'epoch': epoch+1,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'best_acc': best_acc
    }, checkpoint_path)

    print("Checkpoint Saved")

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

print("Accuracy:", 100*correct/total)